# phoneme-entropy — tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suchirsalhan/phoneme/blob/main/examples/usage_example.ipynb)

Walkthrough of the [`phoneme-entropy`](https://pypi.org/project/phoneme-entropy/)
API using the bundled CELEX sample, so it runs anywhere the package is installed.

Pipeline: load data → phoneme frequencies → Dirichlet α → segmental
informativity & prefix entropy → maximum-entropy fit.


In [ ]:
# Install from PyPI (skip if you already have a local editable checkout)
%pip install -q "phoneme-entropy[plot]"


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict


## 1. Load the bundled CELEX sample


In [ ]:
from phoneme_entropy.data import load_sample
sample = load_sample()
sample.head()


## 2. Phoneme unigram frequencies


In [ ]:
phonfreqs = defaultdict(int)
for word in sample["Word"]:
    for p in word.split():
        phonfreqs[p] += 1
phonfreqs = pd.DataFrame(phonfreqs.items(), columns=["Phoneme", "Frequency"])
phonfreqs.head()


## 3. Symmetric-Dirichlet α from the phoneme distribution

`n_boot=0` gives a deterministic point estimate. Increase it for a bootstrap
standard error (seed numpy for reproducibility).


In [ ]:
from phoneme_entropy import estimate_alpha_entropy
estimate_alpha_entropy(phonfreqs["Frequency"], n_boot=0)


### Predicted vs. observed ranks (Beta order statistics)

Requires the `plot` extra (the `plot` extra).


In [ ]:
import matplotlib.pyplot as plt
from phoneme_entropy.dirichlet import plot_ranks
plot_ranks(phonfreqs["Frequency"], alpha=.7, types="Reversed", loglog=False)
plt.show()


## 4. Segmental informativity & prefix entropy

`segment_informativity` = frequency-weighted next-phoneme surprisal.
`phoneme_prefix_entropy` = mean entropy of words still compatible after each phoneme.


In [ ]:
from phoneme_entropy import segment_informativity, phoneme_prefix_entropy

segment_info_type = segment_informativity(sample["Word"])
segment_info_token = segment_informativity(sample["Word"], sample["Frequency"], alpha=1)
info_gain_type = phoneme_prefix_entropy(sample["Word"])
info_gain_token = phoneme_prefix_entropy(sample["Word"], sample["Frequency"], smoothing="ML")
segment_info_type.head()


## 5. Maximum-entropy fit

Form frequency-weighted target expectations of (surprisal, entropy) and fit the
max-ent distribution whose feature expectations match them.


In [ ]:
from phoneme_entropy import compute_maxent_from_matrix

merged = phonfreqs.merge(segment_info_type, on="Phoneme").merge(info_gain_type, on="Phoneme")
p = merged["Frequency"] / merged["Frequency"].sum()
targets = np.array([np.sum(p * merged["Surprisal"]), np.sum(p * merged["Entropy"])])
F = merged[["Surprisal", "Entropy"]].to_numpy()
probs, lambdas = compute_maxent_from_matrix(F, targets)
lambdas


In [ ]:
import seaborn as sns
sns.regplot(x=np.log(p), y=np.log(probs))
plt.xlabel("(log) observed probability")
plt.ylabel("(log) predicted probability")
plt.show()
